In [ ]:
file_location = '/FileStore/base_semana02.csv'
file_type = "csv"

infer_schema = "true"
first_row_is_header = "true"
delimiter = ";"

df = spark.read.format(file_type) \
    .option("inferSchema", infer_schema) \
    .option("header", first_row_is_header) \
    .option("sep", delimiter) \
    .load(file_location)

display(df)

## Exercício

1. Construir estatísticas para as amostras
  - Taxa de Bad
  - Representatividade dos Maus
  - Participação da amostra

2. Escreva uma função que dado um dataframe, um nome de coluna e um vetor de nomes de estatísticas retorne um dicionário com as estatísticas para os valores únicos da coluna

In [ ]:
from pyspark.sql import functions as F
import timeit

# amostras = df.select("amostra").distinct().collect()
amostras = df.select("foto", "amostra").distinct().collect()
# print(amostras)

print(amostras[0])
print(amostras[0].__getitem__("foto"))
print(amostras[0].__getitem__("amostra"))


In [ ]:

start = timeit.timeit()
lista_sucessos = []

aux = df
for amostra in amostras:
    nome_amostra = amostra.__getitem__('amostra')
    if nome_amostra not in lista_sucessos:
        aux = aux.filter(F.col("amostra") == nome_amostra)
        if aux:
            lista_sucessos.append(nome_amostra)
            aux.write.mode("overwrite").parquet(f"/FileStore/amostras_v2/base_semana02_{nome_amostra}")
            aux = aux.filter(
                F.col("amostra") != nome_amostra
            )
        else:
            raise ValueError(f"Impossível processar amostra {nome_amostra}")
end = timeit.timeit()

print(end - start)

Escreva uma função que dado um dataframe, um nome de coluna e um vetor de nomes de estatísticas retorne um dicionário com as estatísticas para os valores únicos da coluna

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame


def fn_sumario(base: DataFrame, nome_coluna: str, vetor_estatisticas: list[str]) -> dict:
  dicionario_resultado: dict = {}
  dicionario_resultado[nome_coluna] = {}
  if nome_coluna not in base.columns:
    raise ValueError("Coluna não está presente")
  for estat in vetor_estatisticas:
    match estat:
      case "min":
        dicionario_resultado[nome_coluna][estat] = base.select(
          nome_coluna
        ).agg(
          F.min(nome_coluna).alias(f"min_{nome_coluna}")
        ).collect()[0].__getitem__(f"min_{nome_coluna}")
      case "max":
        dicionario_resultado[nome_coluna][estat] = 0
      case "median":
        dicionario_resultado[nome_coluna][estat] = 0
      case "avg":
        dicionario_resultado[nome_coluna][estat] = 0
      case "sum":
        dicionario_resultado[nome_coluna][estat] = 0
      case _:
        print("Caso não tratado")
  print(dicionario_resultado)


In [ ]:
fn_sumario(df, "var_1", ["min", "max"])

In [ ]:
from typing import Callable

def fn_sumario_aprimorado(base: DataFrame, nome_coluna: str, vetor_estatisticas: list[Callable]) -> dict:
  dicionario_resultado: dict = {}  
  dicionario_resultado[nome_coluna] = {}
  if nome_coluna not in base.columns:
    raise ValueError("Coluna não está presente")
  for estat in vetor_estatisticas:
    dicionario_resultado[nome_coluna][estat.__str__().split(" ")[1]] = base.select(
          nome_coluna
        ).agg(
          estat(nome_coluna).alias(f"estat_{nome_coluna}")
        ).collect()[0].__getitem__(f"estat_{nome_coluna}")
  return(dicionario_resultado)

In [ ]:
lista_dicionarios: list[dict] = []
vetor_estats = [F.min, F.max, F.median, F.sum, F.mean]

for coluna in df.columns:
    lista_dicionarios.append(fn_sumario_aprimorado(df, coluna, vetor_estats))

In [ ]:
print(lista_dicionarios)

**Exercício 3**:
Crie uma função fn_sumario_aprimorado_v2, que receba uma lista de offsets e calcule estatísticas ao longo do tempo especificado. É necessário passar uma coluna de datas como parâmetro.
- (Dica) Transformar a coluna foto em data

In [ ]:
from pyspark.ml.feature import Imputer

# amostras = df.select("foto", "amostra").distinct().collect()
imputer = Imputer()
imputer.setInputCols(['var_3'])
imputer.setOutputCols(['trat_var_3'])
imputer.setStrategy('mode')
base_tratada = imputer.fit(df)
df = base_tratada.transform(df)


In [ ]:
from pyspark.ml.feature import Bucketizer

bucketizer = Bucketizer()
bucketizer.setSplits([1, 2, 5, 10, 50, 75, float("inf")])
bucketizer.setInputCol('var_1')
bucketizer.setOutputCol('cat_var_1')
bucketizer.setHandleInvalid('keep')

dados_cat = bucketizer.transform(df)
display(dados_cat.select('var_1', 'cat_var_1'))

Exercício 04
- Calcular a taxa de preenchimento para algum dos atributos
- Calcular a taxa de variabilidade para algum dos atributos
- Calcular a correlação entre dois atributos (Crammer e Pearson)
- Aplicar uma escala para uma variável contínua
- Aplicar o método Lasso para selecionar atributos
- Calcular IV
- Aplicação do modelo regressão logística